# 04 — Use Cases: Natural Language Processing

**Learning objectives**
- Understand what NLP/NLU is and why language is hard for machines
- Run sentiment analysis with a pretrained BERT model in 3 lines
- Understand what BERT actually does under the hood
- Fine-tune a BERT model on a custom task: detecting Gen Alpha speech

**Prerequisites:** Notebooks 01, 02, 03

---

## 1. Why is Language Hard?

Images are grids of numbers. Audio is a waveform. Text is different — it's a sequence of **symbols whose meaning depends entirely on context**.

- "The bank was steep" vs "The bank was closed" — same word, opposite meanings
- "Not bad" means good
- "I could eat" means "I'm hungry"
- Sarcasm, irony, idioms, cultural references — none of these have fixed rules

A pixel is a pixel. A word is never just a word.

### NLP vs NLU

**NLP (Natural Language Processing)** — manipulating text: parsing, tokenising, translating, summarising.

**NLU (Natural Language Understanding)** — extracting *meaning*: intent, sentiment, entities, relationships.

The line between them is blurry. Modern transformers do both simultaneously.

### The core challenge: representation

Before a model can reason about text, it needs to convert words into numbers. Early approaches used simple lookup tables (each word → a fixed vector). The problem: "bank" always maps to the same vector regardless of context.

BERT solved this with **contextual embeddings** — the vector for a word changes depending on the words around it.

## 2. BERT — How it Works

**BERT** (Bidirectional Encoder Representations from Transformers, Google 2018) is a language model pretrained on the entire English Wikipedia and BooksCorpus (~3.3B words).

### What it learned

BERT was trained on two self-supervised tasks — no human labels needed:

1. **Masked Language Modelling** — randomly mask 15% of tokens in a sentence and predict them.
   `"The [MASK] sat on the mat"` → predict "cat"

2. **Next Sentence Prediction** — given two sentences, predict whether the second follows the first in the original text.

After pretraining, BERT has developed a deep internal representation of English syntax, semantics, and world knowledge — baked into 110 million parameters.

### Architecture

BERT is a **Transformer encoder**: it reads the entire sentence at once (bidirectional), using **self-attention** to let every token look at every other token simultaneously.

```
Input:  [CLS] The cat sat on the mat [SEP]
         ↓
    12 layers of self-attention
         ↓
Output: one vector per token (768 dimensions each)
        The [CLS] vector summarises the whole sentence
```

### Fine-tuning

For downstream tasks, we add a small classification head on top of the `[CLS]` vector — exactly what we did with ResNet in notebook 03, but for text. The rest of BERT can be frozen or fine-tuned depending on dataset size.

## 3. Sentiment Analysis — Pretrained Pipeline

We'll use **DistilBERT** (a distilled, 40% smaller version of BERT that retains 97% of its performance) already fine-tuned on SST-2 (Stanford Sentiment Treebank). No training required — just load and run.

In [ ]:
from transformers import pipeline

sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1   # CPU; change to 0 for GPU or "mps" for Apple Silicon
)

sentences = [
    "This is an absolutely brilliant piece of work.",
    "I want my two hours back.",
    "Not bad at all, actually.",
    "I guess it was fine.",
    "Worst experience of my life.",
    "I can't believe how good this is.",
    "It wasn't terrible, but it wasn't great either.",
    "Napoléon Bonaparte's Code Civil reshaped the entire legal world.",
]

print(f"{'Sentence':<55} {'Label':<10} {'Score'}")
print("-" * 80)
for s in sentences:
    result = sentiment(s)[0]
    bar = "█" * int(result["score"] * 20)
    print(f"{s[:54]:<55} {result['label']:<10} {result['score']:.3f} {bar}")

That's the whole pipeline: tokenise → embed → attend → classify. Three lines of code on top of 110M parameters trained on billions of words.

Notice the edge cases: *"Not bad at all"* requires understanding negation. *"I guess it was fine"* carries hedged, lukewarm sentiment. These are genuinely hard — and the model handles them correctly because it learned from context, not keyword matching.

---

## 4. Gen Alpha Classifier — Fine-tuning BERT on a Custom Task

Now we build something ourselves. The task: given a sentence, is it **Gen Alpha speak** or not?

Gen Alpha vocabulary includes: *rizz, no cap, bussin, slay, NPC, based, skibidi, sigma, delulu, gyatt, fanum tax, mewing, goated, mid, lowkey, it's giving, understood the assignment, rent free, touch grass, ohio, W/L, bet, ate, cooked...*

We'll hand-label a small dataset, fine-tune DistilBERT on it, and see how well it generalises to new sentences.

In [ ]:
# ── Dataset — hand-labelled ────────────────────────────────────────────────────
# Label 1 = Gen Alpha, Label 0 = Normal

gen_alpha = [
    "no cap that fit is absolutely bussin fr fr",
    "bro really said that? he's such an NPC",
    "she understood the assignment and ate, leaving no crumbs",
    "that's giving main character energy lowkey",
    "rizz check: W or L?",
    "he's so sigma, no cap",
    "bro needs to touch grass immediately",
    "that movie was mid, not gonna lie",
    "she's so delulu it's actually giving main character",
    "skibidi rizz activated, no cap",
    "bro is acting so sussy rn",
    "that's a certified W from me",
    "you're so based for saying that",
    "it's giving ohio energy and I hate it",
    "fanum tax incoming, protect your snacks",
    "he's been living rent free in my head",
    "periodt, she absolutely slayed",
    "bet, I'll be there at eight",
    "that's so goated fr fr",
    "bro really said that? massive L take",
    "she's mewing 24/7 and it shows",
    "ngl that slaps different",
    "vibe check: failed completely",
    "no cap I'm on a sigma grindset",
    "caught him in 4K being sus",
    "that's bussin bussin no cap",
    "lowkey goated behavior ngl",
    "she's got rizz, certified W",
    "bro is so ohio rn it's not even funny",
    "he got fanum taxed and didn't even notice",
    "understood the assignment, she's built different",
    "that's highkey bussin fr",
    "touch grass bro you're cooked",
    "W rizz, absolute sigma behavior",
    "she ate that performance, left zero crumbs",
    "this is giving 2024 energy lowkey",
    "no cap she's built different fr",
    "slay queen, you absolutely cooked",
    "bro really thought he was sigma smh",
    "the fanum tax is real out here no cap",
    "based response, certified W take",
    "that's so delulu I genuinely can't",
    "fr that's goated behavior no cap",
    "she's lowkey built different ngl fr",
    "no cap that's mid at best",
    "bro really said that unironically? massive L",
    "vibe check passed but we're still cooked",
    "caught that in 4K, sussy behavior fr",
    "she's got rizz no cap fr fr",
    "ohio moment detected, evacuate immediately",
    "that meal is bussin, certified W",
    "he's an NPC and doesn't even know it",
    "sigma grindset unlocked fr",
    "rent free in my mental real estate no cap",
    "it's giving delulu main character fr",
    "bro said that with his whole chest, W",
    "she really ate and left no crumbs",
    "lowkey mid but highkey bussin somehow",
    "no cap that's the most goated thing I've seen",
    "touch grass bro you're fully cooked rn",
]

normal = [
    "I went to the grocery store this morning",
    "The weather is quite pleasant today",
    "Could you please pass me the salt?",
    "I need to finish this report by tomorrow morning",
    "The movie was really enjoyable",
    "She did an excellent job on the presentation",
    "I think we should reconsider our approach",
    "The restaurant had very good food",
    "Can you help me with this problem?",
    "I'm going to the gym later this afternoon",
    "The book was interesting but a bit long",
    "We should leave early to avoid the traffic",
    "The meeting has been rescheduled for Thursday",
    "I really enjoyed the concert last night",
    "Please send me the documents when you're ready",
    "The project deadline is next week",
    "I disagree with that assessment",
    "The coffee here is much better than usual",
    "We need to have a serious conversation about this",
    "The train was delayed by twenty minutes",
    "I think this is a reasonable solution",
    "She has been working very hard lately",
    "The presentation went quite well overall",
    "I'll call you back in a few minutes",
    "This is taking longer than expected",
    "The new policy seems fair to me",
    "I really like what you've done with the place",
    "We should probably eat something before leaving",
    "The report is due on Friday afternoon",
    "I appreciate your help with this matter",
    "The weather forecast says rain tomorrow",
    "Can we move the meeting to next week?",
    "I finished reading that book you recommended",
    "The team did a great job this quarter",
    "I'm not sure I understand what you mean",
    "Let's discuss this further at our next meeting",
    "The flight was delayed by two hours",
    "I need to buy groceries on my way home",
    "She has a very interesting perspective on this",
    "The results were better than we expected",
    "I think we're making solid progress",
    "The office will be closed on Monday",
    "He gave a very thoughtful response",
    "I'd like to order the pasta please",
    "We should plan the trip well in advance",
    "The instructions were clear and easy to follow",
    "I've been working on this problem for a while",
    "The performance was absolutely outstanding",
    "Could you please repeat that?",
    "I really enjoyed spending time with your family",
    "The kitchen needs to be cleaned before dinner",
    "I think the blue one looks better",
    "We arrived at the hotel around eight o'clock",
    "The exam results will be posted next week",
    "I'll see you at the conference on Tuesday",
    "The soup was delicious and very warming",
    "Please let me know if you have any questions",
    "I need to renew my driver's licence soon",
    "The children played in the park all afternoon",
    "I believe we can find a better solution",
]

texts  = gen_alpha + normal
labels = [1] * len(gen_alpha) + [0] * len(normal)

print(f"Gen Alpha samples: {len(gen_alpha)}")
print(f"Normal samples:    {len(normal)}")
print(f"Total:             {len(texts)}")
print(f"\nExample Gen Alpha: \"{gen_alpha[0]}\"")
print(f"Example Normal:    \"{normal[0]}\"")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertModel
from sklearn.model_selection import train_test_split
import numpy as np

device = (
    torch.device("mps")  if torch.backends.mps.is_available() else
    torch.device("cuda") if torch.cuda.is_available()         else
    torch.device("cpu")
)
print(f"Device: {device}")

# ── Tokeniser ─────────────────────────────────────────────────────────────────
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

# ── Dataset ───────────────────────────────────────────────────────────────────
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.encodings = tokenizer(texts, truncation=True, padding=True,
                                   max_length=max_len, return_tensors="pt")
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encodings.items()}, self.labels[idx]

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

train_ds = TextDataset(train_texts, train_labels, tokenizer)
val_ds   = TextDataset(val_texts,   val_labels,   tokenizer)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=16)

print(f"Train: {len(train_ds)}  |  Val: {len(val_ds)}")

In [ ]:
# ── Model: DistilBERT + binary classification head ────────────────────────────
class GenAlphaClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        # Freeze all BERT layers — only train the head
        for param in self.bert.parameters():
            param.requires_grad = False
        self.head = nn.Sequential(
            nn.Linear(768, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, input_ids, attention_mask):
        # [CLS] token output = sentence-level representation
        cls_output = self.bert(input_ids=input_ids,
                               attention_mask=attention_mask).last_hidden_state[:, 0, :]
        return self.head(cls_output).squeeze(1)

model = GenAlphaClassifier().to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}  ({100*trainable/total:.2f}%)")
print("\nBERT backbone frozen — training the classification head only.")

In [ ]:
import matplotlib.pyplot as plt

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.head.parameters(), lr=2e-3)

EPOCHS  = 20
history = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(EPOCHS):
    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    for batch, labels in train_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = labels.to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(labels)

    # ── Validate ──────────────────────────────────────────────────────────────
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch, labels in val_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = labels.to(device)
            logits         = model(input_ids, attention_mask)
            val_loss      += criterion(logits, labels).item() * len(labels)
            preds          = (torch.sigmoid(logits) >= 0.5).float()
            correct       += (preds == labels).sum().item()
            total         += len(labels)

    tl = running_loss / len(train_ds)
    vl = val_loss / len(val_ds)
    va = correct / total
    history["train_loss"].append(tl)
    history["val_loss"].append(vl)
    history["val_acc"].append(va)
    print(f"Epoch {epoch+1:02d}/{EPOCHS}  train={tl:.4f}  val={vl:.4f}  acc={va:.3f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ep = range(1, EPOCHS + 1)
ax1.plot(ep, history["train_loss"], label="Train", color="steelblue")
ax1.plot(ep, history["val_loss"],   label="Val",   color="tomato")
ax1.set_title("Loss"); ax1.set_xlabel("Epoch"); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(ep, history["val_acc"], color="seagreen", linewidth=2)
ax2.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, label="Random baseline")
ax2.set_ylim(0, 1.05); ax2.set_title("Val Accuracy")
ax2.set_xlabel("Epoch"); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.suptitle("DistilBERT — Gen Alpha Classifier", fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ── Try your own sentences ────────────────────────────────────────────────────
def classify(sentences):
    model.eval()
    enc = tokenizer(sentences, truncation=True, padding=True,
                    max_length=64, return_tensors="pt").to(device)
    with torch.no_grad():
        probs = torch.sigmoid(model(enc["input_ids"], enc["attention_mask"])).cpu()

    print(f"{'Sentence':<52} {'Result':<14} Confidence")
    print("-" * 85)
    for s, p in zip(sentences, probs):
        label = "Gen Alpha 🔥" if p >= 0.5 else "Normal 📝"
        conf  = p.item() if p >= 0.5 else 1 - p.item()
        print(f"{s[:51]:<52} {label:<14} {conf:.1%}")

classify([
    "no cap that's bussin fr fr",
    "I have a meeting at three o'clock",
    "she understood the assignment lowkey",
    "The quarterly report is due on Friday",
    "bro is so cooked rn no cap",
    "Could you help me with this please?",
    "that's giving sigma energy fr",
    "We should discuss this further tomorrow",
    "rizz check failed, massive L",
    "The soup was absolutely delicious",
])

## Summary

| Concept | What we did |
|---|---|
| **Contextual embeddings** | BERT produces different vectors for the same word depending on context |
| **Pretrained pipeline** | 3 lines of code to run state-of-the-art sentiment analysis |
| **[CLS] token** | DistilBERT's sentence-level representation, used as input to the classifier |
| **Frozen backbone** | Kept BERT's 66M parameters fixed, only trained our 98K-parameter head |
| **Small dataset fine-tuning** | 96 labelled sentences were enough because BERT already understands language |

### Key takeaway

The pattern is identical to notebook 03: take a pretrained backbone (ResNet / BERT), freeze it, attach a small task-specific head, train only the head. The backbone does the heavy lifting — you supply the task definition.

The difference from CV: language models transfer even more aggressively. BERT's internal representations encode grammar, semantics, and world knowledge. A model trained on Wikipedia can detect Gen Alpha slang with fewer than 100 examples.

---

**Next notebook:** Speech — STT (transcription) and TTS (synthesis).